In [1]:
import numpy as np
from scipy.optimize import fsolve

# Data
V1 = 1 + 0j
Z = 0.02 + 0.08j
Y = 1 / Z

Y21 = -Y
Y22 = Y

# Last (negativ injeksjon)
S2 = -0.8 - 0.3j

def equations(x):
    Vr, Vi = x
    V2 = Vr + 1j*Vi

    I2 = Y21*V1 + Y22*V2
    Scalc = V2 * np.conj(I2)

    return [
        Scalc.real - S2.real,
        Scalc.imag - S2.imag
    ]

sol = fsolve(equations, [0.9, -0.1])

V2 = sol[0] + 1j*sol[1]

print("V2 =", V2)
print("|V2| =", abs(V2))
print("angle =", np.degrees(np.angle(V2)))

V2 = (0.9545723264784117-0.058000000000000045j)
|V2| = 0.9563327488266872
angle = -3.477028066132069


In [2]:
import numpy as np
from scipy.optimize import fsolve

# Data
Z12 = 3.4848 + 1j*13.9392
Z13 = 1.7424 + 1j*8.7120
Z23 = 2.6136 + 1j*10.4544

# Base values
Zb = (132**2) / 100  # ohm

# Convert impedances to per unit
z12 = Z12 / Zb
z13 = Z13 / Zb
z23 = Z23 / Zb

# Branch admittances
y12 = 1 / z12
y13 = 1 / z13
y23 = 1 / z23

# Y-bus matrix
Y = np.array([
    [y12+y13,   -y12,      -y13],
    [-y12,      y12+y23,   -y23],
    [-y13,      -y23,      y13+y23]
], dtype=complex)

# Loads (negative injections)
S2 = -(0.8 + 0.3j)
S3 = -(0.5 + 0.2j)

V1 = 1 + 0j

def equations(x):
    V2 = x[0] + 1j*x[1]
    V3 = x[2] + 1j*x[3]

    V = np.array([V1, V2, V3])

    I = Y @ V
    S = np.array([V[k] * np.conj(I[k]) for k in range(3)])

    return [
        S[1].real - S2.real,
        S[1].imag - S2.imag,
        S[2].real - S3.real,
        S[2].imag - S3.imag
    ]

# Solve load flow
sol = fsolve(equations, [0.95, -0.05, 0.97, -0.03])

V2 = sol[0] + 1j*sol[1]
V3 = sol[2] + 1j*sol[3]

V = np.array([V1, V2, V3])

I = Y @ V
S = np.array([V[k] * np.conj(I[k]) for k in range(3)])

print("V2 =", V2)
print("|V2| =", abs(V2))
print("angle(V2) =", np.degrees(np.angle(V2)))

print("V3 =", V3)
print("|V3| =", abs(V3))
print("angle(V3) =", np.degrees(np.angle(V3)))

print("S1 =", S[0])

print("Network losses =", S[0] + S[1] + S[2])

print("\nYbus =")
print(Y)

V2 = (0.9680064731047222-0.043340587818133665j)
|V2| = 0.9689762321775827
angle(V2) = -2.5635938487171646
V3 = (0.9769488269359329-0.03238302450517606j)
|V3| = 0.9774853813370296
angle(V3) = -1.8984940627473814
S1 = (1.31539670615626+0.5676638123928583j)
Network losses = (0.015396706034174201+0.0676638120433598j)

Ybus =
[[ 6.78733032-30.99547511j -2.94117647+11.76470588j
  -3.84615385+19.23076923j]
 [-2.94117647+11.76470588j  6.8627451 -27.45098039j
  -3.92156863+15.68627451j]
 [-3.84615385+19.23076923j -3.92156863+15.68627451j
   7.76772247-34.91704374j]]


# Lab5: Lastflyt i 2-buss og 3-buss systemer ved bruk av Gauss-Seidel og PowerFactory

## Læringsmål

Etter gjennomført lab skal studenten kunne:

- Modellere enkle kraftsystemer i per unit.
- Sette opp en Y-bussmatrise.
- Utføre lastflytberegninger med Gauss-Seidel-metoden.
- Modellere kraftsystemer i PowerFactory.
- Sammenligne analytiske beregninger med simuleringsresultater.

---

# Del 1: Lastflyt i et 2-buss system

## Systembeskrivelse

Et 66 kV regionalnett forsyner et lokalt distribusjonsområde via en transmisjonslinje.

### Data

Slack-buss:

$$
V_1 = 1 \angle 0^\circ \text{ p.u.}
$$

Systembase:

$$
S_{base}=100 \text{ MVA}
$$

Last ved buss 2:

$$
S_2 = 20 + j5 \text{ MVA}
$$

Linjeimpedans:

$$
Z_{12}=0.01+j0.05 \text{ p.u.}
$$

### Enlinjeskjema

```text
                Z12 = 0.01 + j0.05 pu

 Bus 1 -------------------------------- Bus 2
 Slack                                   PQ

V1 = 1∠0° pu                      S2 = 20 + j5 MVA
```

---

## Oppgave 1

Konverter lasten til per unit.

### Løsningsforslag

$$
S_2
=
\frac{20+j5}{100}
=
0.2+j0.05
\text{ p.u.}
$$

Ved lastflyt benyttes negativ injeksjon:

$$
P_2=-0.2
$$

$$
Q_2=-0.05
$$

---

## Oppgave 2

Bestem linjeadmittansen.

### Løsningsforslag

$$
Y_{12}
=
\frac{1}{0.01+j0.05}
$$

Ved kompleks konjugering:

$$
Y_{12}
=
\frac{0.01-j0.05}
     {0.01^2+0.05^2}
$$

$$
Y_{12}
=
3.846-j19.231
$$

---

## Oppgave 3

Sett opp Y-bussmatrisen.

### Løsningsforslag

$$
Y_{bus}
=
\begin{bmatrix}
3.846-j19.231 & -3.846+j19.231\\
-3.846+j19.231 & 3.846-j19.231
\end{bmatrix}
$$

---

## Oppgave 4

Utfør én Gauss-Seidel-iterasjon.

Startgjetning:

$$
V_2^{(0)}
=
1+j0
$$

Gauss-Seidel-formelen blir:

$$
V_2^{(k+1)}
=
\frac{1}{Y_{22}}
\left(
\frac{S_2^*}{(V_2^{(k)})^*}
-
Y_{21}V_1
\right)
$$

Utfør én iterasjon og bestem

$$
V_2^{(1)}
$$

---

## Oppgave 5

Modellér systemet i PowerFactory.

### Modell

- Bus 1: External Grid
- Bus 2: Load
- Element mellom bussene: Common Impedance (ElmZpu)

### Parametre

#### Last

```text
P = 20 MW
Q = 5 MVAr
```

#### Common Impedance

```text
R = 0.01 pu
X = 0.05 pu
```

#### External Grid

```text
U = 1.0 pu
vinkel = 0°
```

---

## Oppgave 6

Kjør lastflyt og registrer:

- Spenning på buss 2
- Fasevinkel på buss 2
- Aktiv effekt fra slack-bussen
- Reaktiv effekt fra slack-bussen
- Linjetap

### Sammenligning

| Variabel | Gauss-Seidel | PowerFactory |
|-----------|-------------|--------------|
| $|V_2|$ | | |
| $\delta_2$ | | |
| $P_{slack}$ | | |
| $Q_{slack}$ | | |

---

# Del 2: Lastflyt i et 3-buss system

## Systembeskrivelse

Distribusjonsområdet utvides med en ny forsyningsstasjon på buss 3.

### Data

Slack-buss:

$$
V_1=1\angle0^\circ
$$

Systembase:

$$
S_{base}=100 \text{ MVA}
$$

Laster:

$$
S_2 = 20 + j5 \text{ MVA}
$$

$$
S_3 = 15 + j4 \text{ MVA}
$$

Linjeimpedanser:

$$
Z_{12}=j0.04
$$

$$
Z_{13}=j0.03
$$

$$
Z_{23}=j0.05
$$

### Enlinjeskjema

```text
                j0.04

         Bus 1 ------- Bus 2
           |             |
           |             |
        j0.03         j0.05
           |             |
           |             |
           ----- Bus 3 ---

         S3 = 15 + j4 MVA
```

---

## Oppgave 1

Konverter alle laster til per unit.

### Løsningsforslag

$$
S_2 = 0.20+j0.05
$$

$$
S_3 = 0.15+j0.04
$$

---

## Oppgave 2

Bestem grenadmittansene.

### Løsningsforslag

$$
Y_{12}
=
\frac{1}{j0.04}
=
-j25
$$

$$
Y_{13}
=
\frac{1}{j0.03}
=
-j33.33
$$

$$
Y_{23}
=
\frac{1}{j0.05}
=
-j20
$$

---

## Oppgave 3

Sett opp Y-bussmatrisen.

Benytt:

$$
Y_{ii}
=
\sum Y_{ik}
$$

og

$$
Y_{ik}
=
-\frac{1}{Z_{ik}}
\qquad
(i \neq k)
$$

---

## Oppgave 4

Utfør én Gauss-Seidel-iterasjon.

Startgjetning:

$$
V_2^{(0)}=1
$$

$$
V_3^{(0)}=1
$$

Beregn:

$$
V_2^{(1)}
$$

Deretter:

$$
V_3^{(1)}
$$

---

## Oppgave 5

Modellér systemet i PowerFactory.

### Busser

```text
Bus 1 → External Grid
Bus 2 → Load
Bus 3 → Load
```

### Common Impedance-elementer

```text
Bus 1 - Bus 2
R = 0
X = 0.04 pu
```

```text
Bus 1 - Bus 3
R = 0
X = 0.03 pu
```

```text
Bus 2 - Bus 3
R = 0
X = 0.05 pu
```

---

## Oppgave 6

Kjør lastflyt og sammenlign med de teoretiske beregningene.

| Variabel | Gauss-Seidel | PowerFactory |
|-----------|-------------|--------------|
| $|V_2|$ | | |
| $|V_3|$ | | |
| $\delta_2$ | | |
| $\delta_3$ | | |

---

# Refleksjonsoppgaver

1. Hvorfor blir spenningen lavere jo lenger bort fra slack-bussen man kommer?

2. Hvorfor blir spenningsvinklene negative ved belastning?

3. Hvordan endres spenningsprofilen dersom lasten dobles?

4. Hvilke fordeler har PowerFactory sammenlignet med manuell Gauss-Seidel-beregning?

5. Hvorfor blir Y-bussmatrisen viktigere etter hvert som kraftsystemet vokser?